# Phase 7 · Part C · Model 3 LoRA and AdaLoRA

| Run | Phase 1 backbone | Adaptation |
|---|---|---|
| LoRA | Model 3 local/global Transformer | Fixed-rank adapters and a trainable retrieval projection |
| AdaLoRA | Model 3 local/global Transformer | Adaptive rank allocation and a trainable retrieval projection |

Each technique has its own checkpoint, local evaluation, prediction file and submission ZIP.


## Required files and environment

- Attach `phase7_part_c_assets.zip` or its extracted contents for the Phase 1 checkpoints and evaluator.
- Attach the official `Phase_7_PCR/` data, or enable Internet and set the `HF_TOKEN` secret to download it.
- Enable a GPU. To resume in a fresh session, restore the run folder and rerun setup before training.

In [5]:
import importlib.metadata
import importlib.util
import subprocess
import sys

missing = [name for name in ('ijson', 'huggingface_hub') if importlib.util.find_spec(name) is None]
if importlib.util.find_spec('sklearn') is None:
    missing.append('scikit-learn')

for name, expected in {'peft': '0.17.1', 'transformers': '4.55.4'}.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(f'{name}=={expected}')

if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


In [6]:
import contextlib
import csv
import gc
import hashlib
import html
import heapq
import importlib.util
import json
import math
import os
import random
import re
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import quote

import ijson
import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, HTML, display
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

PAD, UNK, BOS, EOS = 0, 1, 2, 3
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE, '| GPUs:', torch.cuda.device_count())


PyTorch: 2.11.0+cu130 | device: cuda:0 | GPUs: 2


## Settings

Development Micro-F1@7 selects the checkpoint and ranking policy. Test export uses both together.

In [7]:
VERSION = 'phase7-pcr-final-v2'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase7_pcr_final')
DATA_DIR = None             # Set to the attached Phase_7_PCR folder if needed.
DATA_REVISION = None        # Use the same Hugging Face commit in both notebooks.
MODEL_ASSET_DIRS = {}       # Optional mapping such as {'model1': '/kaggle/input/.../model1_decoder'}.

SEED = 42
SEQUENCE_LENGTH = 512
DOCUMENT_WINDOWS = 3
BATCH_QUERIES = 4
POSITIVES_PER_QUERY = 2
NEGATIVES_PER_QUERY = 3
HARD_NEGATIVES_PER_QUERY = 2
CONTRASTIVE_TEMPERATURE = 0.07
EARLY_STOP_PATIENCE = None  # Set a positive integer to enable patience stopping.
EARLY_STOP_MIN_DELTA = 0.001 # Absolute Micro-F1@7 improvement over the patience reference.
EPOCHS = 25                 # Total epochs, including any completed before resuming.
MAX_UPDATES = None          # Optional total optimizer update limit.
MAX_TRAIN_MINUTES = None    # Optional per-call pause. Final dev evaluation is additional.
SAVE_EVERY = 100

USE_ALL_GPUS = True
RUN_CAPACITY_CHECKS = True
EMBED_BATCH = 8
QUERY_BLOCK = 128
CANDIDATE_BLOCK = 4096

PHASE1_UPLOAD = "phase7_part_c_assets.zip"

ADALORA_ALLOCATION_EPOCHS = 3  # Rank schedule for a new run. Resume keeps the saved schedule.

DISTILL_WEIGHT = 0.5
DISTILL_TEMPERATURE = 2.0

RUNS = [{'name': 'model3_lora', 'method': 'lora', 'model': 'model3', 'embedding_dim': 256, 'lr': 0.0001, 'rank': 8, 'alpha': 16}, {'name': 'model3_adalora', 'method': 'adalora', 'model': 'model3', 'embedding_dim': 256, 'lr': 0.0001, 'rank': 8, 'initial_rank': 12, 'alpha': 16, 'orth_reg_weight': 0.1}]
PHASE1_ASSETS = {'model1': {'folder': 'model1_decoder', 'files': {'best.pt': {'bytes': 479334327, 'sha256': 'cbdbdc8fff1d44d999a857997521135e56c6f92e7b53ec4eb1e72f514e0ce2f2', 'archive_path': 'phase5_model_assets/model1_decoder/best.pt'}, 'tokenizer.json': {'bytes': 975958, 'sha256': 'ce28d26edf4caa9358ffe59b573e362d9c05a90e881957fa0f2d054a5bfed152', 'archive_path': 'phase5_model_assets/model1_decoder/tokenizer.json'}, 'run_config.json': {'bytes': 710, 'sha256': '4c97a89900a29586595c84848a1c722681edea88a45ba8951703a806fc757595', 'archive_path': 'phase5_model_assets/model1_decoder/run_config.json'}}}, 'model2': {'folder': 'model2_encoder_decoder', 'files': {'best.pt': {'bytes': 489533205, 'sha256': 'd4d73182b497b12ee9ff6eca43275d23844c24c5103d7209b4764e38b06a84de', 'archive_path': 'phase5_model_assets/model2_encoder_decoder/best.pt'}, 'tokenizer.json': {'bytes': 975958, 'sha256': 'ce28d26edf4caa9358ffe59b573e362d9c05a90e881957fa0f2d054a5bfed152', 'archive_path': 'phase5_model_assets/model2_encoder_decoder/tokenizer.json'}, 'run_config.json': {'bytes': 616, 'sha256': 'd1445d247e47f25516ab3819475e2bfea10e2bfc3d7275c251be2dca5d3c48cf', 'archive_path': 'phase5_model_assets/model2_encoder_decoder/run_config.json'}}}, 'model3': {'folder': 'model3_local_global', 'files': {'best.pt': {'bytes': 473568441, 'sha256': '50b00a9e89390e080178cbec343c40605ae93d9ccf51185e8cea7a47817022f0', 'archive_path': 'phase5_model_assets/model3_local_global/best.pt'}, 'tokenizer.json': {'bytes': 975958, 'sha256': 'ce28d26edf4caa9358ffe59b573e362d9c05a90e881957fa0f2d054a5bfed152', 'archive_path': 'phase5_model_assets/model3_local_global/tokenizer.json'}, 'run_config.json': {'bytes': 2221, 'sha256': '8d81a3eef2186f442e42839f730c295caf85ee37fbd90c055d10ff8ec2ef33ac', 'archive_path': 'phase5_model_assets/model3_local_global/run_config.json'}}}, 'model4': {'folder': 'model4_gated_cnn', 'files': {'best.pt': {'bytes': 477667321, 'sha256': 'fcab9f52a114bf8b281a219d57cb45e8ff892106c20257d2da45335fcd6d5dbb', 'archive_path': 'phase5_model_assets/model4_gated_cnn/best.pt'}, 'tokenizer.json': {'bytes': 975958, 'sha256': 'ce28d26edf4caa9358ffe59b573e362d9c05a90e881957fa0f2d054a5bfed152', 'archive_path': 'phase5_model_assets/model4_gated_cnn/tokenizer.json'}, 'run_config.json': {'bytes': 764, 'sha256': '8a6bfcd4b838f997f5ce706bef0cdfc0b6412c0279573c1ece25a2f68ab947b5', 'archive_path': 'phase5_model_assets/model4_gated_cnn/run_config.json'}}}}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Original Phase 1 definitions


In [8]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [9]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [10]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [11]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [12]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [13]:
MODEL_CLASSES = dict(model1=DecoderLM, model2=EncoderDecoder, model3=LocalGlobalLM, model4=GatedConvLM)


## Data and format checks

The cache streams the six official JSON files into fixed-size token windows. The audit reports split sizes, truncated and empty documents, missing in-pool positives and candidate overlap. Train queries without a usable positive/negative pair are excluded from optimization. Dev scoring retains the full relevance set, including any gold IDs outside the pool.

References for this setup include [full-text lexical retrieval and self-match filtering in Indian case retrieval](https://arxiv.org/html/2610.07437v2), [lexical hard negatives](https://aclanthology.org/2020.emnlp-main.550/), [matching long legal documents in parts](https://arxiv.org/abs/2201.01614), and [normalized score fusion](https://arxiv.org/abs/2210.11934).


In [14]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 2**20), b''):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':')).encode()).hexdigest()

def save_torch(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [15]:
DATA_FILES = {
    'train': ('Train/train.json', 'Train/train_candidates.json'),
    'dev': ('Dev/dev.json', 'Dev/dev_candidates.json'),
    'test': ('Test/test.json', 'Test/test_candidates.json'),
}


def resolve_data():
    required = [name for pair in DATA_FILES.values() for name in pair]
    if DATA_DIR is not None:
        folder = Path(DATA_DIR)
        missing = [name for name in required if not (folder / name).is_file()]
        if missing:
            raise FileNotFoundError(f'{folder} is missing {missing}')
        return folder
    matches = sorted({p.parent.parent for p in INPUT_ROOT.rglob('train.json')
                      if p.parent.name == 'Train'
                      and all((p.parent.parent / name).is_file() for name in required)})
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f'Set DATA_DIR to one official Phase_7_PCR folder: {matches}')
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    try:
        snapshot = snapshot_download('Exploration-Lab/CS-781-Capstone', repo_type='dataset',
            revision=DATA_REVISION, allow_patterns=[f'Phase_7_PCR/{name}' for name in required],
            token=token, cache_dir=OUTPUT_ROOT / 'hf_cache')
    except Exception as exc:
        raise RuntimeError('Attach official Phase_7_PCR data, or enable Internet and an authorized HF_TOKEN secret.') from exc
    folder = Path(snapshot) / 'Phase_7_PCR'
    missing = [name for name in required if not (folder / name).is_file()]
    if missing:
        raise FileNotFoundError(f'The selected official dataset revision is missing {missing}')
    print('Official data revision:', Path(snapshot).name)
    return folder


def resolve_phase1_asset(model_name):
    spec = PHASE1_ASSETS[model_name]
    expected = spec['files']
    folder = Path(MODEL_ASSET_DIRS[model_name]) if model_name in MODEL_ASSET_DIRS else None
    if folder is None:
        options = sorted({path.parent for path in INPUT_ROOT.rglob('best.pt')
                          if path.parent.name == spec['folder'] and (path.parent / 'tokenizer.json').is_file()})
        for option in options:
            if digest_file(option / 'best.pt') == expected['best.pt']['sha256']:
                folder = option
                break
    if folder is None:
        folder = OUTPUT_ROOT / 'phase1_imports' / spec['folder']
        if not all((folder / name).is_file() for name in expected):
            archives = sorted(INPUT_ROOT.rglob(PHASE1_UPLOAD))
            archive_prefix = 'phase7_model_assets'
            if not archives:
                archives = sorted(INPUT_ROOT.rglob('phase5_kaggle_assets.zip'))
                archive_prefix = 'phase5_model_assets'
            if len(archives) != 1:
                raise FileNotFoundError(f'Attach one {PHASE1_UPLOAD}, or set MODEL_ASSET_DIRS to extracted Phase 1 folders.')
            folder.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive:
                for name, meta in expected.items():
                    target = folder / name
                    temp = target.with_suffix('.partial')
                    member = f'{archive_prefix}/{spec["folder"]}/{name}'
                    with archive.open(member) as source, temp.open('wb') as output:
                        shutil.copyfileobj(source, output, length=8 * 2**20)
                    if digest_file(temp) != meta['sha256']:
                        raise ValueError(f'Invalid Phase 1 asset {member}')
                    temp.replace(target)
    for name, meta in expected.items():
        if digest_file(folder / name) != meta['sha256']:
            raise ValueError(f'Unexpected original Phase 1 asset {folder / name}')
    return folder


def stream_cases(path):
    with path.open('rb') as handle:
        first = handle.read(4096).lstrip()[:1]
        handle.seek(0)
        if first == b'{':
            yield from ijson.kvitems(handle, '')
        elif first == b'[':
            for record in ijson.items(handle, 'item'):
                if not isinstance(record, dict) or 'id' not in record:
                    raise ValueError(f'{path.name} has a list record without an id')
                yield record['id'], record
        else:
            raise ValueError(f'{path.name} must contain a JSON object or array')


def case_id(value):
    if type(value) is int:
        value = str(value)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f'Invalid case ID {value!r}')
    return value


def document_text(record):
    text = record['text'] if isinstance(record, dict) else record
    if isinstance(text, list) and all(isinstance(sentence, str) for sentence in text):
        return '\n'.join(text)
    if isinstance(text, str):
        return text
    raise ValueError('Expected case text as a string or list of sentences')


def encode_case(text, tokenizer, length):
    budget = length - 2
    span = length * 8
    if len(text) <= 3 * span:
        tokens = tokenizer.encode(text).ids
        clipped = len(tokens) > DOCUMENT_WINDOWS * budget
        if not clipped:
            views = [tokens[i:i + budget] for i in range(0, len(tokens), budget)] or [[]]
        else:
            starts = [0, (len(tokens) - budget) // 2, len(tokens) - budget]
            views = [tokens[start:start + budget] for start in starts]
    else:
        middle = (len(text) - span) // 2
        center = tokenizer.encode(text[middle:middle + span]).ids
        start = max(0, (len(center) - budget) // 2)
        views = [tokenizer.encode(text[:span]).ids[:budget], center[start:start + budget],
                 tokenizer.encode(text[-span:]).ids[-budget:]]
        clipped = True
    rows = np.full((DOCUMENT_WINDOWS, length), PAD, dtype='<u2')
    for index, tokens in enumerate(views):
        rows[index, :len(tokens) + 2] = [BOS, *tokens, EOS]
    return rows, clipped


def tokenize_file(source, destination, digest, tokenizer, tokenizer_digest, labeled):
    metadata_path = destination.with_suffix('.json')
    identity = dict(version=VERSION, source=digest, tokenizer=tokenizer_digest,
                    sequence_length=SEQUENCE_LENGTH, document_windows=DOCUMENT_WINDOWS, labeled=labeled)
    def shape(ids):
        return len(ids), DOCUMENT_WINDOWS, SEQUENCE_LENGTH

    if metadata_path.is_file() and destination.is_file():
        metadata = read_json(metadata_path)
        if metadata['identity'] == identity and destination.stat().st_size == 2 * math.prod(shape(metadata['ids'])):
            return metadata, np.memmap(destination, dtype='<u2', mode='r', shape=shape(metadata['ids']))
    destination.parent.mkdir(parents=True, exist_ok=True)
    ids, seen, gold = [], set(), {}
    clipped = empty = blank_labels = 0
    temporary = destination.with_suffix('.partial')
    with temporary.open('wb') as output:
        for raw_id, record in tqdm(stream_cases(source), desc=source.name):
            identifier = case_id(raw_id)
            if identifier in seen:
                raise ValueError(f'Duplicate case ID {identifier!r} in {source}')
            if isinstance(record, dict) and 'id' in record and case_id(record['id']) != identifier:
                raise ValueError(f'Key and embedded ID disagree in {source}: {identifier!r}')
            seen.add(identifier)
            ids.append(identifier)
            text = document_text(record)
            row, was_clipped = encode_case(text, tokenizer, SEQUENCE_LENGTH)
            row.tofile(output)
            clipped += was_clipped
            empty += not text.strip()
            if labeled:
                values = record.get('relevant_candidates') if isinstance(record, dict) else None
                if not isinstance(values, list):
                    raise ValueError(f'{source.name}, {identifier}: expected a relevant_candidates list')
                labels = [value for value in values if isinstance(value, str) and value.strip()]
                blank_labels += len(values) - len(labels)
                if len(set(labels)) != len(labels):
                    raise ValueError(f'{source.name}, {identifier}: duplicate gold candidate ID')
                gold[identifier] = labels
    if not ids:
        raise ValueError(f'Empty data file: {source}')
    temporary.replace(destination)
    metadata = dict(identity=identity, ids=ids, gold=gold, clipped=clipped, empty=empty,
                    ignored_invalid_gold_entries=blank_labels)
    write_json(metadata_path, metadata)
    return metadata, np.memmap(destination, dtype='<u2', mode='r', shape=shape(ids))


def prepare_data(data_root, tokenizer, tokenizer_digest):
    if not 8 <= SEQUENCE_LENGTH <= 1408 or tokenizer.get_vocab_size() > 65536:
        raise ValueError('Sequence length or vocabulary exceeds the shared Phase 1 input limits')
    hashes = {name: digest_file(data_root / name) for pair in DATA_FILES.values() for name in pair}
    dataset_id = object_hash(hashes)
    cache = OUTPUT_ROOT / 'token_cache' / object_hash([VERSION, dataset_id, tokenizer_digest, SEQUENCE_LENGTH])[:20]
    data, audit = {}, {}
    for split, (query_name, candidate_name) in DATA_FILES.items():
        q, query_tokens = tokenize_file(data_root / query_name, cache / f'{split}_queries.bin',
                                        hashes[query_name], tokenizer, tokenizer_digest, split != 'test')
        c, candidate_tokens = tokenize_file(data_root / candidate_name, cache / f'{split}_pool.bin',
                                            hashes[candidate_name], tokenizer, tokenizer_digest, False)
        if len(c['ids']) < 7:
            raise ValueError(f'{split} candidate pool has fewer than seven cases')
        candidate_index = {identifier: index for index, identifier in enumerate(c['ids'])}
        self_candidates = [candidate_index.get(qid, -1) for qid in q['ids']]
        gold_self_matches = sum(qid in q['gold'].get(qid, ()) for qid in q['ids'])
        if gold_self_matches:
            print(f'{split}: {gold_self_matches} gold self-matches. Kept in evaluator truth only.')
        if len(c['ids']) == 7 and any(index >= 0 for index in self_candidates):
            raise ValueError(f'{split} pool has fewer than seven eligible cases after excluding self-matches')
        positives = [frozenset(candidate_index[cid] for cid in q['gold'].get(qid, ()) if cid in candidate_index)
                     .difference({own}) for qid, own in zip(q['ids'], self_candidates)]
        gold_sizes = [len(q['gold'].get(qid, ())) for qid in q['ids']]
        usable = [i for i, labels in enumerate(positives)
                  if 0 < len(labels) < len(c['ids']) - int(self_candidates[i] >= 0)]
        data[split] = SimpleNamespace(query_ids=q['ids'], candidate_ids=c['ids'], gold=q['gold'],
            queries=query_tokens, candidates=candidate_tokens, positives=positives, usable=usable,
            self_candidates=self_candidates)
        audit[split] = dict(queries=len(q['ids']), candidates=len(c['ids']), query_cases_clipped=q['clipped'],
            candidate_cases_clipped=c['clipped'], blank_query_texts=q['empty'], blank_candidate_texts=c['empty'],
            query_candidate_id_overlap=sum(qid in candidate_index for qid in q['ids']),
            gold_pairs=sum(gold_sizes) if split != 'test' else None,
            gold_self_matches=gold_self_matches if split != 'test' else None,
            gold_pairs_outside_pool=sum(cid not in candidate_index for row in q['gold'].values() for cid in row)
                if split != 'test' else None,
            pool_micro_f1_at_7_ceiling=2 * sum(min(7, len(p)) for p in positives) /
                (7 * len(q['ids']) + sum(gold_sizes)) if split != 'test' else None,
            queries_without_in_pool_gold=sum(not p for p in positives) if split != 'test' else None,
            usable_training_queries=len(usable) if split == 'train' else None,
            ignored_invalid_gold_entries=q['ignored_invalid_gold_entries'] if split != 'test' else None)
    if not data['train'].usable or not sum(map(len, data['dev'].gold.values())):
        raise ValueError('The official train/dev files must provide usable relevance labels')
    provenance = dict(dataset_id=dataset_id, file_sha256=hashes, data_path=str(data_root),
                      tokenizer_sha256=tokenizer_digest, sequence_length=SEQUENCE_LENGTH,
                      document_windows=DOCUMENT_WINDOWS, splits=audit)
    write_json(OUTPUT_ROOT / 'data_audit.json', provenance)
    print(json.dumps(audit, indent=2))
    return data, provenance


In [16]:
BASE_FOLDERS = {spec['model']: resolve_phase1_asset(spec['model']) for spec in RUNS}
tokenizer_paths = [folder / 'tokenizer.json' for folder in BASE_FOLDERS.values()]
tokenizer_hashes = {digest_file(path) for path in tokenizer_paths}
if len(tokenizer_hashes) != 1:
    raise ValueError('These models must use the same original Phase 1 tokenizer')
TOKENIZER = ScratchByteBPE.from_file(tokenizer_paths[0])
DATA, AUDIT = prepare_data(resolve_data(), TOKENIZER, tokenizer_hashes.pop())


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Phase_7_PCR/Test/test_candidates.json:   0%|          | 0.00/71.1M [00:00<?, ?B/s]

Phase_7_PCR/Train/train.json:   0%|          | 0.00/33.2M [00:00<?, ?B/s]

Phase_7_PCR/Train/train_candidates.json:   0%|          | 0.00/160M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json:   0%|          | 0.00/51.9M [00:00<?, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

dev_candidates.json: 0it [00:00, ?it/s]

test.json: 0it [00:00, ?it/s]

test_candidates.json: 0it [00:00, ?it/s]

{
  "train": {
    "queries": 827,
    "candidates": 4320,
    "query_cases_clipped": 778,
    "candidate_cases_clipped": 3725,
    "blank_query_texts": 0,
    "blank_candidate_texts": 0,
    "query_candidate_id_overlap": 827,
    "gold_pairs": 5308,
    "gold_self_matches": 0,
    "gold_pairs_outside_pool": 0,
    "pool_micro_f1_at_7_ceiling": 0.6385509597188429,
    "queries_without_in_pool_gold": 10,
    "usable_training_queries": 817,
    "ignored_invalid_gold_entries": 0
  },
  "dev": {
    "queries": 118,
    "candidates": 1023,
    "query_cases_clipped": 115,
    "candidate_cases_clipped": 887,
    "blank_query_texts": 0,
    "blank_candidate_texts": 0,
    "query_candidate_id_overlap": 118,
    "gold_pairs": 973,
    "gold_self_matches": 0,
    "gold_pairs_outside_pool": 0,
    "pool_micro_f1_at_7_ceiling": 0.6181211784324625,
    "queries_without_in_pool_gold": 0,
    "usable_training_queries": null,
    "ignored_invalid_gold_entries": 0
  },
  "test": {
    "queries": 237,
  

## Retrieval, official evaluator, and resumable training


In [17]:
class LowRankLinear(nn.Module):
    def __init__(self, base, method, rank, alpha, dropout):
        super().__init__()
        self.base, self.method = base, method
        base.requires_grad_(False)
        self.scale = alpha / rank
        self.drop = nn.Dropout(dropout)
        columns = base.in_features if method == 'lora' else base.out_features
        self.A = nn.Parameter(base.weight.new_empty(rank, columns))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        if self.method == 'lora':
            return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)
        if self.method == 'lorma':
            z = F.linear(x, self.base.weight)
            z = z + self.scale * F.linear(F.linear(self.drop(z), self.A), self.B)
            return z if self.base.bias is None else z + self.base.bias
        raise ValueError('Unknown adaptation method.')
    def merged(self):
        with torch.no_grad():
            delta = self.B.float() @ self.A.float()
            if self.method == 'lorma':
                delta = delta @ self.base.weight.float()
            self.base.weight.add_((self.scale * delta).to(self.base.weight.dtype))
        return self.base

def autocast_context(device):
    return torch.autocast('cuda', dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()


In [18]:
class IA3Scale(nn.Module):
    """Scale the output of key/value projections or the input to an FFN output projection."""

    def __init__(self, base, on_input=False):
        super().__init__()
        self.base = base.requires_grad_(False)
        self.on_input = on_input
        width = base.in_features if on_input else base.out_features
        self.scale = nn.Parameter(torch.ones(width))

    def forward(self, x):
        if self.on_input:
            return self.base(x * self.scale.to(x.dtype))
        result = self.base(x)
        return result * self.scale.to(result.dtype)


def apply_low_rank(module, method, rank, alpha):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LowRankLinear(child, method, rank, alpha, 0.0))
        else:
            apply_low_rank(child, method, rank, alpha)


def configure_adalora(spec):
    signature_path = OUTPUT_ROOT / 'runs' / spec['name'] / 'run_signature.json'
    if signature_path.is_file():
        spec['allocation_steps'] = read_json(signature_path)['spec']['allocation_steps']
    else:
        updates_per_epoch = math.ceil(len(DATA['train'].usable) / BATCH_QUERIES)
        allocation_epochs = ADALORA_ALLOCATION_EPOCHS
        if EPOCHS is not None:
            allocation_epochs = min(allocation_epochs, EPOCHS)
        spec['allocation_steps'] = updates_per_epoch * allocation_epochs
        if MAX_UPDATES is not None:
            spec['allocation_steps'] = min(spec['allocation_steps'], MAX_UPDATES)
    if spec['allocation_steps'] < 3:
        raise ValueError('AdaLoRA needs at least three planned updates for warmup, allocation and final training.')
    print('AdaLoRA allocation schedule:', spec['allocation_steps'], 'optimizer updates')


class Retriever(nn.Module):
    def __init__(self, config, spec, phase1_weights=None):
        super().__init__()
        self.kind = spec['model']
        self.method = spec['method']
        self.orth_reg_weight = spec.get('orth_reg_weight', 0.0)
        self.base = MODEL_CLASSES[self.kind](**config)
        if phase1_weights is not None:
            self.base.load_state_dict(phase1_weights, strict=True)
        if self.kind == 'model2':
            # Retrieval reads the bidirectional encoder representations.
            del self.base.decoder, self.base.dec_norm
        if self.method in ('lora', 'lorma'):
            self.base.requires_grad_(False)
            apply_low_rank(self.base.blocks, self.method, spec['rank'], spec['alpha'])
        if self.method == 'adalora':
            from peft import AdaLoraConfig
            from peft.tuners.adalora import AdaLoraModel

            targets = [name for name, layer in self.base.named_modules()
                       if name.startswith('blocks.') and isinstance(layer, nn.Linear)]
            updates = spec['allocation_steps']
            adapter = AdaLoraConfig(target_modules=targets, init_r=spec['initial_rank'], target_r=spec['rank'],
                lora_alpha=spec['alpha'], lora_dropout=0.0, total_step=updates,
                tinit=max(1, updates // 10), tfinal=max(1, updates // 5), deltaT=10,
                orth_reg_weight=self.orth_reg_weight)
            self.base = AdaLoraModel(self.base, adapter, 'default')
        if spec['method'] == 'ia3':
            self.base.requires_grad_(False)
            for block in self.base.encoder:
                block.attn.k = IA3Scale(block.attn.k)
                block.attn.v = IA3Scale(block.attn.v)
                block.ff[-1] = IA3Scale(block.ff[-1], on_input=True)
        if spec['method'] == 'distill':
            keep = spec['keep_blocks']
            self.base.blocks = nn.ModuleList([self.base.blocks[i] for i in keep])
            self.base.dilations = [self.base.dilations[i] for i in keep]
            self.base.config['layers'] = len(keep)
            self.base.receptive_field = 1 + (self.base.config['kernel_size'] - 1) * sum(self.base.dilations)
        if self.kind in ('model1', 'model2'):
            self.base.gradient_checkpointing = True
        else:
            self.base.config['activation_checkpointing'] = True
        self.projection = nn.Linear(config['width'], spec['embedding_dim'], bias=False)

    def adapter_penalty(self):
        if self.method != 'adalora':
            return 0.0
        penalties = []
        for name, parameter in self.base.named_parameters():
            if 'lora_A.' in name or 'lora_B.' in name:
                weights = parameter.float()
                gram = weights @ weights.T if 'lora_A.' in name else weights.T @ weights
                identity = torch.eye(gram.shape[0], dtype=gram.dtype, device=gram.device)
                penalties.append(torch.linalg.matrix_norm(gram - identity))
        return self.orth_reg_weight * torch.stack(penalties).mean()

    def update_adapter(self, step):
        if self.method == 'adalora':
            self.base.update_and_allocate(step)

    def adapter_state(self):
        if self.method != 'adalora':
            return None
        allocator = self.base.rankallocator
        return dict(rank_pattern=self.base.peft_config['default'].rank_pattern,
            buffers={name: getattr(allocator, name) for name in ('ipt', 'exp_avg_ipt', 'exp_avg_unc')})

    def restore_adapter_state(self, state, device):
        if self.method != 'adalora':
            return
        if state is None:
            raise ValueError('The AdaLoRA resume checkpoint is missing its rank allocation state.')
        for name, buffers in state['buffers'].items():
            setattr(self.base.rankallocator, name, {key: value.to(device) for key, value in buffers.items()})
        self.base.peft_config['default'].rank_pattern = state['rank_pattern']

    def adapter_ranks(self):
        if self.method == 'adalora':
            pattern = self.base.peft_config['default'].rank_pattern or {}
            return {name: sum(pattern[name]) if name in pattern else parameter.shape[0]
                    for name, parameter in self.base.model.named_parameters() if 'lora_E.' in name}
        return {name: layer.A.shape[0] for name, layer in self.base.named_modules()
                if isinstance(layer, LowRankLinear)}

    def encode_window(self, ids):
        if self.kind == 'model1':
            positions = torch.arange(ids.shape[1], device=ids.device)
            hidden = self.base.drop(self.base.token(ids) + self.base.position(positions))
            for block in self.base.blocks:
                if self.training and self.base.gradient_checkpointing:
                    hidden = checkpoint(lambda x, layer=block: layer(x)[0], hidden, use_reentrant=False)
                else:
                    hidden = block(hidden)[0]
            hidden = self.base.final_norm(hidden)
        elif self.kind == 'model2':
            hidden = self.base.encode(ids)[0]
        else:
            hidden = self.base.hidden(ids)[0]
        mask = ids.ne(PAD).unsqueeze(-1)
        pooled = (hidden.float() * mask).sum(1) / mask.sum(1).clamp_min(1)
        return F.normalize(self.projection(pooled).float(), dim=-1)

    def forward(self, ids):
        if ids.ndim == 2:
            return self.encode_window(ids)
        rows, windows, length = ids.shape
        flat = ids.reshape(rows * windows, length)
        active = flat[:, 0].ne(PAD).nonzero(as_tuple=True)[0]
        vectors = self.encode_window(flat.index_select(0, active))
        summed = vectors.new_zeros((rows, vectors.shape[1])).index_add(0, active // windows, vectors)
        return F.normalize(summed, dim=-1)


def initialize_retriever(spec):
    folder = BASE_FOLDERS[spec['model']]
    saved = torch.load(folder / 'best.pt', map_location='cpu', weights_only=True)
    config = saved['model_config']
    if config['vocab_size'] != TOKENIZER.get_vocab_size():
        raise ValueError('Checkpoint vocabulary differs from the verified Phase 1 tokenizer')
    if SEQUENCE_LENGTH > config.get('context', 1408):
        raise ValueError('SEQUENCE_LENGTH exceeds this checkpoint context')
    model = Retriever(config, spec, saved['model'])
    return model, config


def wrap_model(model):
    model = model.to(DEVICE)
    if DEVICE.type == 'cuda' and USE_ALL_GPUS and torch.cuda.device_count() > 1:
        return nn.DataParallel(model)
    return model


def parameter_counts(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad))


def training_batch(data, query_indices, rng):
    candidate_set = set()
    for q in query_indices:
        gold = data.positives[q]
        candidate_set.update(rng.sample(sorted(gold), min(POSITIVES_PER_QUERY, len(gold))))
        mined = data.hard_negatives[q]
        negatives = set(rng.sample(mined, min(HARD_NEGATIVES_PER_QUERY, len(mined))))
        forbidden = gold.union(negatives, {data.self_candidates[q]})
        remaining = min(NEGATIVES_PER_QUERY - len(negatives),
                        len(data.candidate_ids) - len(forbidden - {-1}))
        if len(forbidden) > len(data.candidate_ids) // 2:
            eligible = [i for i in range(len(data.candidate_ids)) if i not in forbidden]
            negatives.update(rng.sample(eligible, remaining))
        else:
            target = len(negatives) + remaining
            while len(negatives) < target:
                candidate = rng.randrange(len(data.candidate_ids))
                if candidate not in forbidden:
                    negatives.add(candidate)
        candidate_set.update(negatives)
    candidates = sorted(candidate_set)
    positive_mask = torch.tensor([[c in data.positives[q] for c in candidates] for q in query_indices],
                                 dtype=torch.bool, device=DEVICE)
    allowed = torch.tensor([[c != data.self_candidates[q] for c in candidates] for q in query_indices],
                           dtype=torch.bool, device=DEVICE)
    tokens = np.concatenate((data.queries[query_indices], data.candidates[candidates]))
    return torch.as_tensor(tokens.astype(np.int64), device=DEVICE), candidates, positive_mask, allowed


def batch_losses(embeddings, positive_mask, allowed, teacher_scores=None):
    count = positive_mask.shape[0]
    scores = embeddings[:count] @ embeddings[count:].T / CONTRASTIVE_TEMPERATURE
    scores = scores.masked_fill(~allowed, -1e4)
    mean_positive = scores.masked_fill(~positive_mask, 0).sum(1) / positive_mask.sum(1)
    supervised = (scores.logsumexp(1) - mean_positive).mean()
    if teacher_scores is None:
        return supervised, supervised, None
    temperature = DISTILL_TEMPERATURE
    soft = F.kl_div(F.log_softmax(scores / temperature, dim=1),
                    F.softmax(teacher_scores.masked_fill(~allowed, -1e4) / temperature, dim=1),
                    reduction='batchmean') * temperature**2
    total = (1 - DISTILL_WEIGHT) * supervised + DISTILL_WEIGHT * soft
    return total, supervised, soft


In [19]:
EVALUATOR_PATH = OUTPUT_ROOT / "evaluate_phase7.py"
EVALUATOR_PATH.write_text('#!/usr/bin/env python3\n"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).\n\nIt uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /\nMicro-Recall@7 definitions as the Codabench scorer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nK = 7\n\n\ndef reject_constant(value: str) -> None:\n    raise ValueError(f"Non-standard JSON numeric value \'{value}\' is not allowed.")\n\n\ndef pairs_no_duplicates(pairs):\n    result = {}\n    duplicates = []\n    for key, value in pairs:\n        if key in result:\n            duplicates.append(key)\n        result[key] = value\n    if duplicates:\n        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")\n    return result\n\n\ndef load_json(path: Path) -> Any:\n    try:\n        with path.open("r", encoding="utf-8") as f:\n            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n    except json.JSONDecodeError as exc:\n        raise ValueError(\n            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"\n        ) from exc\n\n\ndef load_truth(path: Path) -> dict[str, list[str]]:\n    data = load_json(path)\n    # Accept either the simple gold.json mapping or a Phase-7 dev query file:\n    # {query_id: {text: [...], relevant_candidates: [...]}}\n    if not isinstance(data, dict) or not data:\n        raise ValueError("Truth must be a non-empty JSON object.")\n    out: dict[str, list[str]] = {}\n    for query_id, value in data.items():\n        if isinstance(value, dict) and "relevant_candidates" in value:\n            labels = value["relevant_candidates"]\n        else:\n            labels = value\n        if not isinstance(query_id, str) or not query_id.strip():\n            raise ValueError("Truth contains an empty/invalid query ID.")\n        if not isinstance(labels, list):\n            raise ValueError(f"Truth for query \'{query_id}\' must contain a list of relevant candidate IDs.")\n        cleaned=[]; seen=set()\n        for idx,candidate_id in enumerate(labels):\n            if not isinstance(candidate_id,str) or not candidate_id.strip():\n                # Treat blank source labels as absent labels, matching the builder.\n                continue\n            if candidate_id in seen:\n                raise ValueError(f"Truth for query \'{query_id}\' contains duplicate candidate ID \'{candidate_id}\'.")\n            seen.add(candidate_id); cleaned.append(candidate_id)\n        out[query_id]=cleaned\n    return out\n\n\ndef load_candidate_ids(path: Path) -> set[str]:\n    data = load_json(path)\n    if not isinstance(data, list) or not data:\n        raise ValueError("candidate_ids must be a non-empty JSON list.")\n    ids=[]; seen=set()\n    for idx,v in enumerate(data):\n        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")\n        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID \'{v}\'.")\n        seen.add(v); ids.append(v)\n    return set(ids)\n\n\ndef load_prediction(path: Path) -> Any:\n    return load_json(path)\n\n\ndef validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:\n    errors=[]\n    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]\n    gold_ids=set(gold); pred_ids=set(pred)\n    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID \'{q}\'.")\n    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID \'{q}\'.")\n    for q in sorted(gold_ids & pred_ids):\n        vals=pred[q]\n        if not isinstance(vals,list):\n            errors.append(f"Query \'{q}\': prediction must be a list."); continue\n        if len(vals)!=K: errors.append(f"Query \'{q}\': expected exactly {K} candidates, got {len(vals)}.")\n        seen=set()\n        for i,c in enumerate(vals):\n            if not isinstance(c,str) or not c.strip(): errors.append(f"Query \'{q}\': index {i} is not a non-empty string."); continue\n            if c in seen: errors.append(f"Query \'{q}\': duplicate candidate ID \'{c}\'.")\n            seen.add(c)\n            if c not in candidates: errors.append(f"Query \'{q}\': candidate ID \'{c}\' is outside the candidate pool.")\n    return errors\n\n\ndef score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:\n    tp=fp=fn=0\n    for q,labels in gold.items():\n        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)\n    precision=tp/(tp+fp) if tp+fp else 0.0\n    recall=tp/(tp+fn) if tp+fn else 0.0\n    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0\n    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}\n\n\ndef parse_args():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")\n    p.add_argument("--prediction",type=Path,required=True)\n    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")\n    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")\n    return p.parse_args()\n\n\ndef main():\n    a=parse_args()\n    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)\n    errs=validate(pred,gold,candidates)\n    if errs:\n        print("LOCAL VALIDATION FAILED")\n        for i,e in enumerate(errs,1): print(f"  {i}. {e}")\n        print(f"Total validation errors: {len(errs)}")\n        raise SystemExit(1)\n    result=score(gold,pred)\n    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}\n    print("Validation passed.")\n    print(f"Micro-F1@7: {scores[\'primary_metric_1\']:.10f}")\n    print(f"Micro-Precision@7: {scores[\'secondary_metric_1\']:.10f}")\n    print(f"Micro-Recall@7: {scores[\'secondary_metric_2\']:.10f}")\n    print(f"TP={result[\'tp\']} FP={result[\'fp\']} FN={result[\'fn\']}")\n    if a.output:\n        a.output.parent.mkdir(parents=True,exist_ok=True)\n        with a.output.open(\'w\',encoding=\'utf-8\') as f: json.dump(scores,f,indent=2); f.write(\'\\n\')\n        print(f"Wrote {a.output}")\n\nif __name__==\'__main__\': main()\n', encoding="utf-8")
evaluator_spec = importlib.util.spec_from_file_location("official_phase7_evaluator", EVALUATOR_PATH)
EVALUATOR = importlib.util.module_from_spec(evaluator_spec)
evaluator_spec.loader.exec_module(EVALUATOR)


In [20]:
def show_download(path):
    path = Path(path).resolve()
    for module_name in ('jupyter_server.serverapp', 'notebook.notebookapp'):
        try:
            for server in importlib.import_module(module_name).list_running_servers():
                root = server.get('root_dir') or server.get('notebook_dir')
                if not root:
                    continue
                try:
                    relative = path.relative_to(Path(root).resolve()).as_posix()
                except ValueError:
                    continue
                url = server.get('base_url', '/').rstrip('/') + '/files/' + quote(relative, safe='/') + '?download=1'
                display(HTML(f'<a href="{html.escape(url, quote=True)}" download="{html.escape(path.name, quote=True)}" target="_blank">Download {html.escape(path.name)}</a>'))
                return
        except (ImportError, OSError, RuntimeError):
            continue
    display(FileLink(os.path.relpath(path, Path.cwd())))


In [21]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

RANKING_POLICIES = ('dense', 'hybrid_tfidf', 'hybrid_lexical')


def normalized_scores(scores, data):
    scores = np.array(scores, dtype=np.float32, copy=True)
    rows = np.flatnonzero(np.asarray(data.self_candidates) >= 0)
    own = np.asarray(data.self_candidates)[rows]
    scores[rows, own] = np.inf
    low = scores.min(axis=1, keepdims=True)
    scores[rows, own] = -np.inf
    high = scores.max(axis=1, keepdims=True)
    scores -= low
    scores /= np.where(high > low, high - low, 1)
    scores[rows, own] = 0
    return scores


def rank_scores(scores, data):
    scores = np.array(scores, dtype=np.float32, copy=True)
    rows = np.flatnonzero(np.asarray(data.self_candidates) >= 0)
    scores[rows, np.asarray(data.self_candidates)[rows]] = -np.inf
    indices = np.argsort(-scores, axis=1, kind='stable')[:, :7]
    prediction = {qid: [data.candidate_ids[index] for index in row]
                  for qid, row in zip(data.query_ids, indices)}
    expected = data.gold or dict.fromkeys(data.query_ids, [])
    errors = EVALUATOR.validate(prediction, expected, set(data.candidate_ids))
    if errors:
        raise ValueError('\n'.join(errors[:20]))
    return prediction


def lexical_scores(split):
    data = DATA[split]
    query_file, pool_file = [Path(AUDIT['data_path']) / name for name in DATA_FILES[split]]
    path = OUTPUT_ROOT / 'lexical_cache' / object_hash([VERSION, AUDIT['dataset_id'], split])
    path = path.with_suffix('.npz')
    shape = len(data.query_ids), len(data.candidate_ids)
    if path.is_file():
        with np.load(path, allow_pickle=False) as cache:
            scores = {name: cache[name] for name in ('tfidf', 'bm25')}
        if all(matrix.shape == shape and np.isfinite(matrix).all() for matrix in scores.values()):
            return scores

    def texts(source, expected_ids):
        for (raw_id, record), expected in zip(stream_cases(source), expected_ids, strict=True):
            if case_id(raw_id) != expected:
                raise ValueError(f'Case order changed in {source}')
            yield document_text(record)

    print(f'Full-text lexical index: {split}', flush=True)
    vectorizer = CountVectorizer(stop_words='english', max_features=100_000, dtype=np.float32)
    candidates = vectorizer.fit_transform(texts(pool_file, data.candidate_ids))
    queries = vectorizer.transform(texts(query_file, data.query_ids))
    tfidf = TfidfTransformer(sublinear_tf=True)
    candidate_tfidf = tfidf.fit_transform(candidates).T.tocsr()
    query_tfidf = tfidf.transform(queries)
    df = np.bincount(candidates.indices, minlength=candidates.shape[1])
    idf = np.log1p((len(data.candidate_ids) - df + 0.5) / (df + 0.5))
    lengths = np.asarray(candidates.sum(axis=1)).ravel()
    norm = 1.5 * (0.25 + 0.75 * lengths / lengths.mean())
    bm25 = candidates.copy()
    bm25.data *= 2.5 / (bm25.data + np.repeat(norm, np.diff(bm25.indptr)))
    bm25.data *= idf[bm25.indices]
    candidate_bm25 = bm25.T.tocsr()
    query_terms = queries.copy()
    query_terms.data.fill(1)
    scores = {name: np.empty(shape, dtype=np.float32) for name in ('tfidf', 'bm25')}
    for start in range(0, shape[0], QUERY_BLOCK):
        block = slice(start, start + QUERY_BLOCK)
        scores['tfidf'][block] = (query_tfidf[block] @ candidate_tfidf).toarray()
        scores['bm25'][block] = (query_terms[block] @ candidate_bm25).toarray()
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.with_suffix('.partial').open('wb') as handle:
        np.savez(handle, **scores)
    path.with_suffix('.partial').replace(path)
    return scores


def lexical_features(scores, data):
    tfidf = normalized_scores(scores['tfidf'], data)
    bm25 = normalized_scores(scores['bm25'], data)
    return dict(tfidf=tfidf, bm25=bm25, mixture=0.75 * tfidf + 0.25 * bm25)


def prepare_lexical():
    data = DATA['train']
    scores = lexical_features(lexical_scores('train'), data)['mixture']
    # Keep a small pool of plausible negatives for each query and resample during training.
    data.hard_negatives = []
    for row, gold, own in zip(scores, data.positives, data.self_candidates):
        row[list(gold)] = -np.inf
        if own >= 0:
            row[own] = -np.inf
        order = np.argsort(-row, kind='stable')[:64]
        data.hard_negatives.append([int(index) for index in order if row[index] > 0])
    write_json(OUTPUT_ROOT / 'train_hard_negatives.json', {
        qid: [data.candidate_ids[index] for index in row]
        for qid, row in zip(data.query_ids, data.hard_negatives)})

    dev = DATA['dev']
    features = lexical_features(lexical_scores('dev'), dev)
    predictions = {name: rank_scores(features[name], dev) for name in ('tfidf', 'bm25', 'mixture')}
    scores = {name: EVALUATOR.score(dev.gold, ranking) for name, ranking in predictions.items()}
    write_json(OUTPUT_ROOT / 'dev_lexical_predictions.json', predictions)
    write_json(OUTPUT_ROOT / 'dev_lexical_baselines.json', scores)
    print({name: round(result['primary_metric_1'], 6) for name, result in scores.items()}, flush=True)
    return features


In [22]:
@torch.inference_mode()
def embed_cases(model, tokens):
    model.eval()
    output = None
    for start in tqdm(range(0, len(tokens), EMBED_BATCH), desc='Embedding cases', leave=False):
        end = min(start + EMBED_BATCH, len(tokens))
        ids = torch.as_tensor(np.asarray(tokens[start:end], dtype=np.int64), device=DEVICE)
        with autocast_context(DEVICE):
            encoded = model(ids)
        if not torch.isfinite(encoded).all().item():
            raise FloatingPointError('A case embedding contains a non-finite value')
        if output is None:
            output = torch.empty((len(tokens), encoded.shape[1]), dtype=torch.float16)
        output[start:end] = encoded.to(device='cpu', dtype=torch.float16)
    return output


@torch.inference_mode()
def dense_scores(query_embeddings, candidate_embeddings):
    scores = np.empty((len(query_embeddings), len(candidate_embeddings)), dtype=np.float32)
    for start in tqdm(range(0, len(query_embeddings), QUERY_BLOCK), desc='Ranking', leave=False):
        queries = query_embeddings[start:start + QUERY_BLOCK].to(DEVICE, torch.float32)
        for offset in range(0, len(candidate_embeddings), CANDIDATE_BLOCK):
            candidates = candidate_embeddings[offset:offset + CANDIDATE_BLOCK].to(DEVICE, torch.float32)
            scores[start:start + len(queries), offset:offset + len(candidates)] = (queries @ candidates.T).cpu().numpy()
    return scores


def predict_split(model, data, features, policies):
    start = time.perf_counter()
    queries = embed_cases(model, data.queries)
    candidates = embed_cases(model, data.candidates)
    embedding_seconds = time.perf_counter() - start
    dense = normalized_scores(dense_scores(queries, candidates), data)
    scores = {'dense': dense}
    if 'hybrid_tfidf' in policies:
        scores['hybrid_tfidf'] = 0.75 * features['tfidf'] + 0.25 * dense
    if 'hybrid_lexical' in policies:
        scores['hybrid_lexical'] = 0.75 * features['mixture'] + 0.25 * dense
    prediction = {policy: rank_scores(scores[policy], data) for policy in policies}
    return prediction, dict(embedding_seconds=embedding_seconds,
        ranking_seconds=time.perf_counter() - start - embedding_seconds)


def save_dev_evidence(folder, prediction, data, metrics, timing):
    write_json(folder / 'dev_predictions.json', prediction)
    write_json(folder / 'dev_candidate_ids.json', data.candidate_ids)
    write_json(folder / 'dev_gold.json', data.gold)
    write_json(folder / 'dev_metrics.json', dict(metrics=metrics, timing=timing))
    with (folder / 'dev_errors.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['query_id', 'hits', 'gold_count', 'missed_gold_ids', 'false_positive_ids'])
        for qid, ranked in prediction.items():
            gold = set(data.gold[qid])
            writer.writerow([qid, len(gold.intersection(ranked)), len(gold),
                             ' '.join(sorted(gold.difference(ranked))),
                             ' '.join(cid for cid in ranked if cid not in gold)])


def zip_files(path, files):
    temporary = path.with_suffix('.partial')
    path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for source, member in files:
            archive.write(source, member)
    with zipfile.ZipFile(temporary) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise ValueError(f'Damaged ZIP entry: {bad_member}')
    temporary.replace(path)
    return path


def load_run_model(path):
    payload = torch.load(path, map_location='cpu', weights_only=True)
    model = Retriever(payload['model_config'], payload['spec'])
    model.load_state_dict(payload['model'], strict=True)
    return model, payload


def create_submission(spec):
    folder = OUTPUT_ROOT / 'runs' / spec['name']
    model, payload = load_run_model(folder / 'best.pt')
    signature = payload['signature']
    if (payload['spec'] != spec or signature['dataset_id'] != AUDIT['dataset_id']
            or signature['version'] != VERSION or signature['tokenizer_sha256'] != AUDIT['tokenizer_sha256']):
        raise ValueError('Best checkpoint does not match the current experiment or official files')
    if signature['sequence_length'] != SEQUENCE_LENGTH or signature['document_windows'] != DOCUMENT_WINDOWS:
        raise ValueError('Best checkpoint used different document windows')
    policy = payload['ranking_policy']
    data = DATA['test']
    features = lexical_features(lexical_scores('test'), data) if policy != 'dense' else None
    predictor = wrap_model(model)
    predictions, timing = predict_split(predictor, data, features, (policy,))
    prediction = predictions[policy]
    write_json(folder / 'prediction.json', prediction)
    write_json(folder / 'test_timing.json', dict(timing, ranking_policy=policy))
    del predictor, model, payload
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    archive = zip_files(folder / 'submission.zip', [(folder / 'prediction.json', 'prediction.json')])
    print(f'{spec["name"]}: validated {len(prediction)} test queries with {policy}')
    show_download(archive)
    return archive


In [23]:
def memory_report():
    if DEVICE.type != 'cuda':
        return []
    count = torch.cuda.device_count() if USE_ALL_GPUS else 1
    return [dict(device=torch.cuda.get_device_name(i),
                 peak_allocated_mib=round(torch.cuda.max_memory_allocated(i) / 2**20, 1),
                 peak_reserved_mib=round(torch.cuda.max_memory_reserved(i) / 2**20, 1),
                 capacity_mib=round(torch.cuda.get_device_properties(i).total_memory / 2**20, 1))
            for i in range(count)]


def reset_memory_peaks():
    for index, _ in enumerate(memory_report()):
        torch.cuda.reset_peak_memory_stats(index)


def capacity_check(spec):
    seed_everything(SEED)
    reset_memory_peaks()
    model, _ = initialize_retriever(spec)
    net = wrap_model(model)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=spec['lr'])
    rows = BATCH_QUERIES * (1 + POSITIVES_PER_QUERY + NEGATIVES_PER_QUERY)
    ids = torch.randint(4, TOKENIZER.get_vocab_size(), (rows, DOCUMENT_WINDOWS, SEQUENCE_LENGTH), device=DEVICE)
    mask = torch.eye(BATCH_QUERIES, rows - BATCH_QUERIES, dtype=torch.bool, device=DEVICE)
    start = time.perf_counter()
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda', init_scale=1024.0)
        with autocast_context(DEVICE):
            embeddings = net(ids)
        loss, _, _ = batch_losses(embeddings, mask, torch.ones_like(mask))
        loss = loss + model.adapter_penalty()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        scaler.step(optimizer)
        scaler.update()
        model.update_adapter(1)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        report = dict(parameters=parameter_counts(model), maximum_case_rows=rows, windows_per_case=DOCUMENT_WINDOWS,
                      seconds=time.perf_counter() - start, gpu_memory=memory_report())
        print(json.dumps(report, indent=2))
        write_json(OUTPUT_ROOT / 'runs' / spec['name'] / 'capacity.json', report)
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Lower BATCH_QUERIES before creating this run and repeat the capacity check.') from exc
    finally:
        del optimizer, net, model
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()


def experiment_signature(spec, teacher):
    return dict(version=VERSION, spec=spec, dataset_id=AUDIT['dataset_id'], sequence_length=SEQUENCE_LENGTH,
        document_windows=DOCUMENT_WINDOWS, ranking_policies=list(RANKING_POLICIES),
        tokenizer_sha256=AUDIT['tokenizer_sha256'], phase1_sha256=PHASE1_ASSETS[spec['model']]['files']['best.pt']['sha256'],
        seed=SEED, batch_queries=BATCH_QUERIES, negatives_per_query=NEGATIVES_PER_QUERY,
        positives_per_query=POSITIVES_PER_QUERY, hard_negatives_per_query=HARD_NEGATIVES_PER_QUERY,
        contrastive_temperature=CONTRASTIVE_TEMPERATURE,
        teacher_sha256=teacher['sha256'] if teacher is not None else None,
        distill_temperature=DISTILL_TEMPERATURE if teacher is not None else None,
        distill_weight=DISTILL_WEIGHT if teacher is not None else None)


def train_run(spec, teacher=None):
    if (spec['method'] == 'distill') != (teacher is not None):
        raise ValueError('The distillation run requires its locked teacher embeddings')
    if min(BATCH_QUERIES, POSITIVES_PER_QUERY, NEGATIVES_PER_QUERY, SAVE_EVERY) < 1:
        raise ValueError('Training sizes and checkpoint interval must be positive')
    if not 0 <= HARD_NEGATIVES_PER_QUERY <= NEGATIVES_PER_QUERY:
        raise ValueError('Hard negatives must fit within NEGATIVES_PER_QUERY')
    patience_enabled = EARLY_STOP_PATIENCE is not None
    if patience_enabled and (type(EARLY_STOP_PATIENCE) is not int or EARLY_STOP_PATIENCE < 1):
        raise ValueError('EARLY_STOP_PATIENCE must be a positive integer or None')
    if not math.isfinite(EARLY_STOP_MIN_DELTA) or EARLY_STOP_MIN_DELTA < 0:
        raise ValueError('EARLY_STOP_MIN_DELTA must be finite and non-negative')
    if any(limit is not None and limit <= 0 for limit in (EPOCHS, MAX_UPDATES, MAX_TRAIN_MINUTES)):
        raise ValueError('Optional training limits must be positive or None')
    if CONTRASTIVE_TEMPERATURE <= 0 or (teacher is not None and not (DISTILL_TEMPERATURE > 0 and 0 < DISTILL_WEIGHT < 1)):
        raise ValueError('Invalid training temperature or distillation weight')
    folder = OUTPUT_ROOT / 'runs' / spec['name']
    folder.mkdir(parents=True, exist_ok=True)
    signature = experiment_signature(spec, teacher)
    signature_path = folder / 'run_signature.json'
    if signature_path.is_file() and read_json(signature_path) != signature:
        raise ValueError(f'{folder} has another recipe. Restore its settings or use a different run name.')
    write_json(signature_path, signature)
    seed_everything(SEED)
    latest = folder / 'latest.pt'
    resume = None
    if latest.is_file():
        model, resume = load_run_model(latest)
        if resume['signature'] != signature:
            raise ValueError('Resume checkpoint signature mismatch')
        config = resume['model_config']
    else:
        model, config = initialize_retriever(spec)
    counts = parameter_counts(model)
    if teacher is not None and counts['total'] >= teacher['parameters']['total']:
        raise ValueError('The student must have fewer parameters than its teacher')
    reset_memory_peaks()
    net = wrap_model(model)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=spec['lr'], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda', init_scale=1024.0)
    progress = dict(epoch=0, offset=0, step=0, train_seconds=0.0)
    if resume is not None:
        optimizer.load_state_dict(resume['optimizer'])
        scaler.load_state_dict(resume['scaler'])
        model.restore_adapter_state(resume.get('adapter'), DEVICE)
        progress = resume['progress']
        del resume
        print('Resuming at update', progress['step'])
    dev = DATA['dev']
    train = DATA['train']
    summary_file = folder / 'best_summary.json'
    best = read_json(summary_file) if summary_file.is_file() else None
    if best is not None and best['signature'] != signature:
        raise ValueError('Best-result metadata has another recipe')
    scored_file = folder / 'last_scored.json'
    last_scored = read_json(scored_file) if scored_file.is_file() else {}
    if last_scored.get('step', -1) > progress['step']:
        last_scored = {}
    scored_step = last_scored.get('step', -1)
    early = last_scored.get('early_stopping', {})
    if not patience_enabled or early.get('min_delta') != EARLY_STOP_MIN_DELTA:
        early = dict(min_delta=EARLY_STOP_MIN_DELTA, reference_f1=None, stale_epochs=0)
    session_start = time.perf_counter()

    def stop_limit():
        if patience_enabled and early['stale_epochs'] >= EARLY_STOP_PATIENCE:
            return 'early_stopping'
        if EPOCHS is not None and progress['epoch'] >= EPOCHS:
            return 'epoch_limit'
        if MAX_UPDATES is not None and progress['step'] >= MAX_UPDATES:
            return 'update_limit'
        if MAX_TRAIN_MINUTES is not None and (time.perf_counter() - session_start) / 60 >= MAX_TRAIN_MINUTES:
            return 'time_limit'
        return None

    def payload():
        return dict(version=VERSION, signature=signature, spec=spec, model_config=config,
                    model=model.state_dict(), progress=progress.copy(), parameters=counts)

    def save_latest():
        save_torch(latest, dict(payload(), optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                               adapter=model.adapter_state()))

    def score_checkpoint(completed_epoch):
        nonlocal best, scored_step
        save_latest()
        training_memory = memory_report()
        predictions, timing = predict_split(net, dev, DEV_LEXICAL, RANKING_POLICIES)
        policy_metrics = {name: EVALUATOR.score(dev.gold, pred) for name, pred in predictions.items()}
        policy = max(RANKING_POLICIES, key=lambda name: policy_metrics[name]['primary_metric_1'])
        metrics = policy_metrics[policy]
        score = metrics['primary_metric_1']
        counts_for_patience = patience_enabled and completed_epoch and progress['step'] >= spec.get('allocation_steps', 0)
        if counts_for_patience:
            reference = early['reference_f1']
            if reference is None or (score > reference and score >= reference + EARLY_STOP_MIN_DELTA):
                early.update(reference_f1=score, stale_epochs=0)
            else:
                early['stale_epochs'] += 1
        ranks = model.adapter_ranks()
        result = dict(spec=spec, signature=signature, metrics=metrics, step=progress['step'],
                      ranking_policy=policy, policy_metrics=policy_metrics, dense_metrics=policy_metrics['dense'],
                      parameters=counts, gpu_memory=training_memory, dev_timing=timing,
                      train_seconds_at_checkpoint=progress['train_seconds'], adapter_ranks=ranks,
                      epoch=progress['epoch'], completed_epoch=completed_epoch,
                      counts_for_patience=counts_for_patience, early_stopping=early.copy())
        with (folder / 'dev_history.jsonl').open('a', encoding='utf-8') as log:
            log.write(json.dumps(result) + '\n')
        message = (f'{spec["name"]} epoch {progress["epoch"]}, update {progress["step"]}: '
                   f'dev Micro-F1@7 {score:.6f} ({policy}), dense {policy_metrics["dense"]["primary_metric_1"]:.6f}')
        if patience_enabled:
            message += f' | patience {early["stale_epochs"]}/{EARLY_STOP_PATIENCE}'
            if not counts_for_patience:
                message += ' (patience unchanged)'
        print(message, flush=True)
        if best is None or score > best['metrics']['primary_metric_1'] or best['step'] == progress['step']:
            save_torch(folder / 'best.pt', dict(payload(), metrics=metrics, adapter_ranks=ranks,
                ranking_policy=policy, dense_metrics=policy_metrics['dense']))
            result['checkpoint_sha256'] = digest_file(folder / 'best.pt')
            write_json(summary_file, result)
            save_dev_evidence(folder, predictions[policy], dev, metrics, timing)
            write_json(folder / 'dev_policy_predictions.json', predictions)
            best = result
        scored_step = progress['step']
        write_json(scored_file, dict(step=scored_step, metrics=metrics, early_stopping=early))
        net.train()

    net.train()
    # If an interruption occurred during epoch-end scoring, finish it before training more.
    if progress['epoch'] > 0 and progress['offset'] == 0 and scored_step != progress['step']:
        score_checkpoint(completed_epoch=True)
    while stop_limit() is None:
        order = train.usable.copy()
        random.Random(SEED + progress['epoch']).shuffle(order)
        while progress['offset'] < len(order) and stop_limit() is None:
            update_start = time.perf_counter()
            next_offset = min(progress['offset'] + BATCH_QUERIES, len(order))
            query_indices = order[progress['offset']:next_offset]
            update_seed = SEED + progress['step']
            torch.manual_seed(update_seed)
            tokens, candidates, positive_mask, allowed = training_batch(train, query_indices, random.Random(update_seed))
            teacher_scores = None
            if teacher is not None:
                query_vectors = teacher['queries'][query_indices].to(DEVICE, torch.float32)
                candidate_vectors = teacher['candidates'][candidates].to(DEVICE, torch.float32)
                teacher_scores = query_vectors @ candidate_vectors.T / CONTRASTIVE_TEMPERATURE
            for attempt in range(8):
                optimizer.zero_grad(set_to_none=True)
                with autocast_context(DEVICE):
                    embeddings = net(tokens)
                loss, supervised, soft = batch_losses(embeddings, positive_mask, allowed, teacher_scores)
                regularization = model.adapter_penalty()
                loss = loss + regularization
                if not torch.isfinite(loss).item():
                    raise FloatingPointError('Non-finite retrieval loss')
                previous_scale = scaler.get_scale()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=not scaler.is_enabled())
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous_scale:
                    model.update_adapter(progress['step'] + 1)
                    break
            else:
                raise FloatingPointError('Eight AMP retries failed at the same update. Resume from latest.pt.')
            progress['offset'] = next_offset
            progress['step'] += 1
            row = dict(step=progress['step'], epoch=progress['epoch'] + 1, loss=loss.item(),
                       supervised_loss=supervised.item(), distill_loss=soft.item() if soft is not None else None,
                       amp_retries=attempt)
            if model.method == 'adalora':
                row['orthogonality_loss'] = regularization.item()
            progress['train_seconds'] += time.perf_counter() - update_start
            with (folder / 'train_history.jsonl').open('a', encoding='utf-8') as log:
                log.write(json.dumps(row) + '\n')
            if progress['step'] == 1 or progress['step'] % 25 == 0:
                print(f'{spec["name"]}: epoch {row["epoch"]}, update {progress["step"]}, loss {row["loss"]:.4f}', flush=True)
            if progress['step'] % SAVE_EVERY == 0:
                save_latest()
        if progress['offset'] == len(order):
            progress['epoch'] += 1
            progress['offset'] = 0
            score_checkpoint(completed_epoch=True)
        else:
            break
    if progress['step'] == 0:
        raise RuntimeError('No optimizer update completed within the training budget')
    if scored_step != progress['step'] or best is None:
        score_checkpoint(completed_epoch=False)
    stop_reason = stop_limit()
    write_json(folder / 'progress.json', dict(progress, stop_reason=stop_reason, early_stopping=early))
    print(f'{spec["name"]} stopped at epoch {progress["epoch"]}, update {progress["step"]} ({stop_reason}).', flush=True)
    del net, model, optimizer, params
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return best


def summarize_runs(specs, name):
    rows = []
    for spec in specs:
        folder = OUTPUT_ROOT / 'runs' / spec['name']
        report = read_json(folder / 'best_summary.json')
        metrics = report['metrics']
        rows.append(dict(run=spec['name'], method=spec['method'], model=spec['model'],
            ranking_policy=report['ranking_policy'], dense_dev_f1_at_7=report['dense_metrics']['primary_metric_1'],
            dev_f1_at_7=metrics['primary_metric_1'], dev_precision_at_7=metrics['secondary_metric_1'],
            dev_recall_at_7=metrics['secondary_metric_2'], best_update=report['step'],
            parameters=report['parameters']['total'], trainable=report['parameters']['trainable'],
            best_checkpoint_mib=round((folder / 'best.pt').stat().st_size / 2**20, 1),
            training_seconds_at_checkpoint=report['train_seconds_at_checkpoint'],
            dev_embed_seconds=report['dev_timing']['embedding_seconds'],
            dev_rank_seconds=report['dev_timing']['ranking_seconds'],
            teacher_sha256=report['signature']['teacher_sha256'],
            adapter_rank_sum=sum(report.get('adapter_ranks', {}).values()) or None))
    table = pd.DataFrame(rows)
    table.to_csv(OUTPUT_ROOT / f'{name}_comparison.csv', index=False)
    display(table)
    return table

In [24]:
DEV_LEXICAL = prepare_lexical()


Full-text lexical index: train
Full-text lexical index: dev
{'tfidf': 0.355753, 'bm25': 0.132296, 'mixture': 0.330183}


## Adapter settings

Both runs start independently from the same original Model 3 checkpoint and target the same linear layers. LoRA uses fixed rank 8. AdaLoRA starts at rank 12 and reallocates toward an average rank of 8 using PEFT’s importance scheduler and orthogonality penalty. New runs schedule allocation over the first three epochs by default. If enabled, patience begins after the allocation schedule finishes. A resumed run keeps its original schedule and allocator state. The comparison reports the allocated rank at the selected dev checkpoint. If a run pauses during allocation, it may still be above the target budget.

These are additional experiments. Use Parts A and B for the four techniques listed in the Phase 7 instructions. [AdaLoRA paper](https://arxiv.org/abs/2303.10512) · [PEFT AdaLoRA documentation](https://huggingface.co/docs/peft/en/package_reference/adalora)


In [25]:
configure_adalora(RUNS[1])


AdaLoRA allocation schedule: 615 optimizer updates


## Experiment 1 · `model3_lora`

The capacity probe runs a disposable update at the configured maximum batch shape. After it passes, train the retriever and then export its validated test submission.


In [26]:
if RUN_CAPACITY_CHECKS:
    capacity_check(RUNS[0])


{
  "parameters": {
    "total": 119996800,
    "trainable": 1622016
  },
  "maximum_case_rows": 24,
  "windows_per_case": 3,
  "seconds": 4.00848431999998,
  "gpu_memory": [
    {
      "device": "Tesla T4",
      "peak_allocated_mib": 2018.5,
      "peak_reserved_mib": 2532.0,
      "capacity_mib": 14911.7
    },
    {
      "device": "Tesla T4",
      "peak_allocated_mib": 1916.3,
      "peak_reserved_mib": 2366.0,
      "capacity_mib": 14911.7
    }
  ]
}


In [27]:
best_1 = train_run(RUNS[0])


model3_lora: epoch 1, update 1, loss 1.9175
model3_lora: epoch 1, update 25, loss 1.8673
model3_lora: epoch 1, update 50, loss 2.2669
model3_lora: epoch 1, update 75, loss 1.9780
model3_lora: epoch 1, update 100, loss 1.2180
model3_lora: epoch 1, update 125, loss 1.4539
model3_lora: epoch 1, update 150, loss 1.5704
model3_lora: epoch 1, update 175, loss 1.4844
model3_lora: epoch 1, update 200, loss 1.7527


Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/128 [00:00<?, ?it/s]

Ranking:   0%|          | 0/1 [00:00<?, ?it/s]

model3_lora epoch 1, update 205: dev Micro-F1@7 0.363535 (hybrid_tfidf), dense 0.306837
model3_lora: epoch 2, update 225, loss 1.1610
model3_lora: epoch 2, update 250, loss 1.6985
model3_lora: epoch 2, update 275, loss 1.9277
model3_lora: epoch 2, update 300, loss 1.1196
model3_lora: epoch 2, update 325, loss 0.8712
model3_lora: epoch 2, update 350, loss 1.5634
model3_lora: epoch 2, update 375, loss 1.3299
model3_lora: epoch 2, update 400, loss 1.5272


Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

... (91 outputs left out)

Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/128 [00:00<?, ?it/s]

Ranking:   0%|          | 0/1 [00:00<?, ?it/s]

model3_lora epoch 25, update 5125: dev Micro-F1@7 0.372429 (hybrid_tfidf), dense 0.284603
model3_lora stopped at epoch 25, update 5125 (epoch_limit).


In [28]:
submission_1 = create_submission(RUNS[0])


Full-text lexical index: test


Embedding cases:   0%|          | 0/30 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/216 [00:00<?, ?it/s]

Ranking:   0%|          | 0/2 [00:00<?, ?it/s]

model3_lora: validated 237 test queries with hybrid_tfidf


## Experiment 2 · `model3_adalora`

The capacity probe runs a disposable update at the configured maximum batch shape. After it passes, train the retriever and then export its validated test submission.


In [29]:
if RUN_CAPACITY_CHECKS:
    capacity_check(RUNS[1])


I0000 00:00:1794002423.365363      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


{
  "parameters": {
    "total": 120718528,
    "trainable": 2343680
  },
  "maximum_case_rows": 24,
  "windows_per_case": 3,
  "seconds": 2.370550326999364,
  "gpu_memory": [
    {
      "device": "Tesla T4",
      "peak_allocated_mib": 2361.8,
      "peak_reserved_mib": 2662.0,
      "capacity_mib": 14911.7
    },
    {
      "device": "Tesla T4",
      "peak_allocated_mib": 2248.9,
      "peak_reserved_mib": 2546.0,
      "capacity_mib": 14911.7
    }
  ]
}


In [30]:
best_2 = train_run(RUNS[1])


model3_adalora: epoch 1, update 1, loss 2.1822
model3_adalora: epoch 1, update 25, loss 2.0336
model3_adalora: epoch 1, update 50, loss 2.5397
model3_adalora: epoch 1, update 75, loss 2.1366
model3_adalora: epoch 1, update 100, loss 1.3818
model3_adalora: epoch 1, update 125, loss 1.5501
model3_adalora: epoch 1, update 150, loss 1.7152
model3_adalora: epoch 1, update 175, loss 1.6428
model3_adalora: epoch 1, update 200, loss 1.7820


Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/128 [00:00<?, ?it/s]

Ranking:   0%|          | 0/1 [00:00<?, ?it/s]

model3_adalora epoch 1, update 205: dev Micro-F1@7 0.361312 (hybrid_tfidf), dense 0.279044
model3_adalora: epoch 2, update 225, loss 1.2476
model3_adalora: epoch 2, update 250, loss 1.7062
model3_adalora: epoch 2, update 275, loss 1.9965
model3_adalora: epoch 2, update 300, loss 1.1961
model3_adalora: epoch 2, update 325, loss 1.0294
model3_adalora: epoch 2, update 350, loss 1.5399
model3_adalora: epoch 2, update 375, loss 1.3713
model3_adalora: epoch 2, update 400, loss 1.6521


Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

... (91 outputs left out)

Embedding cases:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/128 [00:00<?, ?it/s]

Ranking:   0%|          | 0/1 [00:00<?, ?it/s]

model3_adalora epoch 25, update 5125: dev Micro-F1@7 0.370206 (hybrid_tfidf), dense 0.289049
model3_adalora stopped at epoch 25, update 5125 (epoch_limit).


In [31]:
submission_2 = create_submission(RUNS[1])


Embedding cases:   0%|          | 0/30 [00:00<?, ?it/s]

Embedding cases:   0%|          | 0/216 [00:00<?, ?it/s]

Ranking:   0%|          | 0/2 [00:00<?, ?it/s]

model3_adalora: validated 237 test queries with hybrid_tfidf


## Results and report evidence

Each `runs/` folder has the best checkpoint and its dev predictions, evaluator truth/candidate ID files, per-query errors, ranking comparisons, timing and parameter counts, plus `latest.pt` for resuming. Check `data_audit.json` for context coverage. Inspect missed cases and compare quality, memory, and timing in your report. Upload the two `submission.zip` files separately to Codabench.


In [32]:
comparison = summarize_runs(RUNS, 'part_c')


,run,method,model,ranking_policy,dense_dev_f1_at_7,dev_f1_at_7,dev_precision_at_7,dev_recall_at_7,best_update,parameters,trainable,best_checkpoint_mib,training_seconds_at_checkpoint,dev_embed_seconds,dev_rank_seconds,teacher_sha256,adapter_rank_sum
0,model3_lora,lora,model3,hybrid_tfidf,0.296832,0.382435,0.416465,0.353546,1845,119996800,1622016,457.9,3314.290881,29.623788,0.036365,None,512
1,model3_adalora,adalora,model3,hybrid_tfidf,0.289049,0.377988,0.411622,0.349435,4715,120718528,2343680,460.7,11863.301706,39.533009,0.036933,None,512
